# Study 9: Parameter Estimation (Grid Search)
current iteration: search over length_scale and mu_0

Tests whether fitting the GP `length_scale` and prior mean `mu_0` significantly improves fit\nto participant prevalence ratings, relative to a null model (large lengthscale, fitted mu_0).

`mu_0` sets the population-level baseline: P(z'=1) = sigmoid(mu_0) applies uniformly to all features.\n`length_scale` controls how far GP influence spreads across the feature embedding space.\n\nFor each `(length_scale, mu_0)` pair, runs the full pipeline:
1. Joint MCMC over `(y_train, y'_1, ..., y'_J)`
2. Beta mixture fitting → `E[p' | u, θ]` per test feature
3. Pearson r against empirical participant ratings


**Scientific question**: does feature-space proximity (lengthscale) explain variance in human\nprevalence judgments beyond a flat population baseline (mu_0)?

In [9]:
import sys, csv, pickle as pkl
sys.path.insert(0, ".")

import numpy as np
import jax
import jax.numpy as jnp
import blackjax
import matplotlib.pyplot as plt

from model_jax import (
    make_log_density_fn_joint,
    fit_beta_mixtures_all_features,
    beta_mixture_log_likelihood,
)

In [10]:
with open('../features/set2_features_dataframe.pkl', 'rb') as f:
    df = pkl.load(f)

with open('../../data/study9.csv') as f:
    rows = list(csv.reader(f))
CSV_HEADER = rows[0]
data_rows  = rows[3:]

print(f"Features: {len(df)} total, {(df.split=='train').sum()} train, {(df.split=='test').sum()} test")
print(f"Participants: {len(data_rows)}")

Features: 60 total, 45 train, 15 test
Participants: 402


In [11]:
CSV_TO_FEATURE = {
    'diet_can_eat_spicy_1':  'can eat spicy food',
    'diet_breakfast_late_1': 'eat breakfast very late',
    'diet_five_meals_day_1': 'eat five meals a day',
    'diet_like_juice_pulp_1':'like juice with pulp',
    'diet_pepper_on_all_1':  'put pepper on all their foods',
    'pers_cry_easily_1':     'cry easily',
    'pers_collect_rocks_1':  'like to collect rocks',
    'pers_like_to_dance_1':  'like to dance',
    'pers_like_highfive_1':  'like to give high-fives',
    'pers_read_books_1':     'like to read books',
    'phys_can_roll_tongue_1':'can roll their tongue',
    'phys_can_snap_toes_1':  'can snap with their toes',
    'phys_can_wiggle_ears_1':'can wiggle their ears',
    'phys_cold_hands_feet_1':'have cold hands and feet',
    'phys_snore_sleep_1':    'snore when they sleep',
}
TEST_CSV_COLS = list(CSV_TO_FEATURE.keys())
feat_idx = df.set_index('feature')

train_df = df[df.split == 'train']
x_train  = jnp.array(train_df[['x_2d', 'y_2d']].values)          # (n_train, 2)
u_train  = jnp.zeros(len(train_df), dtype=jnp.int32)              # all generic

test_feature_names = [CSV_TO_FEATURE[c] for c in TEST_CSV_COLS]
x_test = jnp.array(
    [feat_idx.loc[name, ['x_2d', 'y_2d']].values for name in test_feature_names]
)  # (J, 2)

print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")

x_train: (45, 2), x_test: (15, 2)


In [12]:
col_indices = [CSV_HEADER.index(c) for c in TEST_CSV_COLS]

ratings = []
for row in data_rows:
    try:
        vals = [int(row[i]) / 100.0 for i in col_indices]
        ratings.append(vals)
    except (ValueError, IndexError):
        pass

responses = jnp.array(ratings)  # (N, J)
N, J = responses.shape
print(f"responses: {responses.shape}  (N={N} participants, J={J} features)")

empirical_mean = np.array(responses.mean(axis=0))  # (J,)

responses: (402, 15)  (N=402 participants, J=15 features)


In [13]:
LENGTH_SCALES = [0.1, 0.2, 0.4, 0.8, 1.5, 3.0, 10.0]
MU_0_VALS     = [-1.0, -0.5, 0.0, 0.5, 1.0]   # sigmoid: ~0.27, 0.38, 0.50, 0.62, 0.73

# null model: large lengthscale (features are independent), mu_0 free
# full model: best (ls, mu_0) pair

FIXED_PARAMS = {
    'output_scale': 1.5,
    'beta':         3.0,
}

print(f"Grid: {len(LENGTH_SCALES)} lengthscales × {len(MU_0_VALS)} mu_0 values = {len(LENGTH_SCALES)*len(MU_0_VALS)} runs")

Grid: 7 lengthscales × 5 mu_0 values = 35 runs


In [ ]:
from itertools import product
from scipy.special import logsumexp as scipy_logsumexp

n_warmup  = 500
n_samples = 2000
step      = 10   # thin MCMC chain: 1999 → ~200 samples for Option 3

results = []  # one dict per (length_scale, mu_0)
total   = len(LENGTH_SCALES) * len(MU_0_VALS)

for i, (ls, mu_0) in enumerate(product(LENGTH_SCALES, MU_0_VALS)):
    print(f"[{i+1:2d}/{total}] length_scale={ls:<5}  mu_0={mu_0:<5}", end="  ")
    params = {**FIXED_PARAMS, 'length_scale': ls, 'mu_0': mu_0}

    log_density_fn = make_log_density_fn_joint(u_train, x_train, x_test, params)

    init_position = {
        'training_coherences': jnp.zeros(x_train.shape[0]),
        'test_coherences':     jnp.zeros(J),
    }

    rng_key = jax.random.PRNGKey(0)  # same seed every run for comparability
    rng_key, warmup_key = jax.random.split(rng_key)
    warmup = blackjax.window_adaptation(blackjax.nuts, log_density_fn)
    (state, nuts_params), _ = warmup.run(warmup_key, init_position, num_steps=n_warmup)

    nuts = blackjax.nuts(log_density_fn, **nuts_params)

    @jax.jit
    def one_step(state, key):
        return nuts.step(key, state)

    keys = jax.random.split(rng_key, n_samples)
    test_coherences_all = []
    for key in keys[:-1]:
        state, _ = one_step(state, key)
        test_coherences_all.append(np.array(state.position['test_coherences']))
    test_coherences_all = np.array(test_coherences_all)  # (S, J)

    pz1_test = np.array([
        float(np.mean(jax.nn.sigmoid(jnp.array(test_coherences_all[:, j]))))
        for j in range(J)
    ])  # (J,) posterior mean P(z'=1), used for pred_prevalence only

    # Option 3: proper Monte Carlo integral over y' posterior
    # log P(ratings | u, θ) ≈ logsumexp_s [ log P(ratings | pz1_s) ] - log S_eff
    log_liks_s = []
    for s in range(0, test_coherences_all.shape[0], step):
        pz1_s    = jnp.array(jax.nn.sigmoid(test_coherences_all[s]))  # (J,)
        pz1_s_bc = jnp.tile(pz1_s, (N, 1))                            # (N, J)
        beta_params_s = fit_beta_mixtures_all_features(responses, pz1_s_bc)
        log_liks_s.append(beta_mixture_log_likelihood(responses, pz1_s, beta_params_s))

    log_lik = float(scipy_logsumexp(log_liks_s) - np.log(len(log_liks_s)))
    print(f"log_lik = {log_lik:.1f}")

    results.append({
        'length_scale': ls,
        'mu_0':         mu_0,
        'pz1_test':     pz1_test,
        'log_lik':      log_lik,
    })

print("\nGrid search complete.")

In [ ]:
ll_grid = np.array([res['log_lik'] for res in results]).reshape(len(LENGTH_SCALES), len(MU_0_VALS))

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(ll_grid, aspect='auto', cmap='viridis', vmin=ll_grid.min(), vmax=ll_grid.max())
ax.set_xticks(range(len(MU_0_VALS)))
ax.set_xticklabels([f"{m}\n(σ={1/(1+np.exp(-m)):.2f})" for m in MU_0_VALS], fontsize=8)
ax.set_yticks(range(len(LENGTH_SCALES)))
ax.set_yticklabels(LENGTH_SCALES)
ax.set_xlabel("mu_0  (baseline P(z'=1) = sigmoid(mu_0))")
ax.set_ylabel('length_scale')
ax.set_title('Log likelihood: model vs. human prevalence ratings')
plt.colorbar(im, ax=ax, label='Log likelihood')
for i in range(len(LENGTH_SCALES)):
    for j in range(len(MU_0_VALS)):
        ax.text(j, i, format(ll_grid[i, j], '.0f'), ha='center', va='center',
                fontsize=7, color='white' if ll_grid[i, j] < ll_grid.mean() else 'black')
plt.tight_layout()
plt.show()

best_ll = max(results, key=lambda r: r['log_lik'])
null_results = [r for r in results if r['length_scale'] == LENGTH_SCALES[-1]]
null_best_ll = max(null_results, key=lambda r: r['log_lik'])

print(f"Best:  length_scale={best_ll['length_scale']},  mu_0={best_ll['mu_0']},  log_lik={best_ll['log_lik']:.1f}")
print(f"Null:  length_scale={null_best_ll['length_scale']}, mu_0={null_best_ll['mu_0']}, log_lik={null_best_ll['log_lik']:.1f}")